# 🛣️ RoadScan-MA — [RoadScan-MA1.ipynb]
**Auteurs :** Mohamed Amine Belasri & Yahya Amajane  
**ENSAM Meknès — Projet académique IATD 2026** 
**Description :** Ce notebook retrace le pipeline complet d'entraînement pour RoadScan-MA. Il inclut le traitement des données géospatiales, le pseudo-labeling des dégradations routières, l'entraînement itératif de modèles YOLOv8 (Teachers) et la classification finale.

## 1. Setup
Installation des dépendances, montage de l'environnement Google Drive et importation des librairies principales.

In [ ]:
# Importation et montage de Google Drive pour accéder aux datasets
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive monté !")

# Installation du framework Ultralytics pour YOLOv8
!pip install ultralytics -q
import ultralytics
ultralytics.checks()
print("✅ Ultralytics prêt !")

# Installations supplémentaires pour la cartographie et les métadonnées
!pip install gpxpy folium opencv-python-headless pillow -q
!apt-get install -y libimage-exiftool-perl -q

# Importations globales
import os, shutil, random, gc, json, time, base64, io, cv2
import numpy as np
from pathlib import Path
from datetime import datetime, timezone, timedelta
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image

## 2. Data (Préparation, Pseudo-Labeling et Traitement GPS)
Cette section regroupe toute la logique de préparation des données : la fusion des datasets (D1, D2, D4, D5), la création des crops, le pseudo-labeling par vote de modèles, et l'extraction des coordonnées GPS depuis les vidéos brutes.

In [ ]:
# ===================================================================
# 2.1 PRÉPARATION DU DATASET D4 (TEACHER)
# ===================================================================
d4_img = "/content/drive/MyDrive/RoadScan-MA/D4_severity/images"
d4_lbl = "/content/drive/MyDrive/RoadScan-MA/D4_severity/labels"

base = "/content/D4_teacher"
for split in ["train", "val"]:
    os.makedirs(f"{base}/{split}/images", exist_ok=True)
    os.makedirs(f"{base}/{split}/labels", exist_ok=True)

images = [f for f in os.listdir(d4_img) if f.endswith(('.jpg','.png'))]
random.shuffle(images)
split_idx  = int(len(images) * 0.8)
train_imgs = images[:split_idx]
val_imgs   = images[split_idx:]

# Répartition des images en Train/Val
for split, imgs in [("train", train_imgs), ("val", val_imgs)]:
    for img_file in imgs:
        shutil.copy(f"{d4_img}/{img_file}", f"{base}/{split}/images/{img_file}")
        lbl_file = img_file.replace('.jpg','.txt').replace('.png','.txt')
        if os.path.exists(f"{d4_lbl}/{lbl_file}"):
            shutil.copy(f"{d4_lbl}/{lbl_file}", f"{base}/{split}/labels/{lbl_file}")

print(f"✅ Train : {len(train_imgs)} images")
print(f"✅ Val   : {len(val_imgs)} images")

# Création du fichier YAML pour l'entraînement D4
yaml_content = """path: /content/D4_teacher
train: train/images
val: val/images

nc: 3
names:
  0: minor_pothole
  1: medium_pothole
  2: major_pothole
"""
with open("/content/D4_teacher/d4_teacher.yaml", "w") as f:
    f.write(yaml_content)
print("✅ YAML créé !")

# Vérification finale des dossiers
for split in ["train", "val"]:
    imgs_len = len(os.listdir(f"{base}/{split}/images"))
    lbls_len = len(os.listdir(f"{base}/{split}/labels"))
    print(f"✅ {split} → {imgs_len} images / {lbls_len} labels")

In [ ]:
# ===================================================================
# 2.2 COLLECTE ET PSEUDO-LABELING DES NIDS-DE-POULE (D1, D2, D5)
# ===================================================================
paths = {
    "D1": {
        "img": "/content/drive/MyDrive/RoadScan-MA/D1_RDD2022_India/India/train/images",
        "lbl": "/content/drive/MyDrive/RoadScan-MA/D1_RDD2022_India/India/train/labels",
        "pothole_class": 3  # D40
    },
    "D2": {
        "img": "/content/drive/MyDrive/RoadScan-MA/D2_lorenzoarcioni/data/images_cropped",
        "lbl": "/content/drive/MyDrive/RoadScan-MA/D2_lorenzoarcioni/data/labels_cropped",
        "pothole_class": 0  # Pothole original
    },
    "D5": {
        "img": "/content/drive/MyDrive/RoadScan-MA/D5_roboflow",
        "lbl": None,
        "pothole_class": 0
    }
}

pothole_imgs = []
for ds_name, ds in paths.items():
    if ds["lbl"] is None:
        for root, dirs, files in os.walk(ds["img"]):
            for f in files:
                if f.endswith(('.jpg','.png')):
                    pothole_imgs.append(f"{root}/{f}")
    else:
        for lbl_file in os.listdir(ds["lbl"]):
            if not lbl_file.endswith('.txt'): continue
            with open(f"{ds['lbl']}/{lbl_file}") as f:
                classes = [int(l.split()[0]) for l in f if l.strip()]
            if ds["pothole_class"] in classes:
                img_file = lbl_file.replace('.txt','.jpg')
                img_path = f"{ds['img']}/{img_file}"
                if os.path.exists(img_path):
                    pothole_imgs.append(img_path)

print(f"✅ {len(pothole_imgs)} images pothole trouvées !")

os.makedirs("/content/pothole_imgs", exist_ok=True)
for i, src in enumerate(pothole_imgs):
    dst = f"/content/pothole_imgs/{Path(src).name}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)
print("✅ Copie locale terminée !")

In [ ]:
# ===================================================================
# 2.3 EXTRACTION GPS ET SYNCHRONISATION VIDÉO (Meknès)
# ===================================================================
import gpxpy

shutil.copy('/content/drive/MyDrive/RoadScan-MA/Raw_videos/Video2.MOV', '/content/Video2.MOV')
!exiftool -GPS* -Location* -ee /content/Video2.MOV

VIDEO_PATH = '/content/Video2.MOV'
GPX_PATH   = '/content/drive/MyDrive/RoadScan-MA/Raw_videos/2 take shot.gpx'
OUTPUT_DIR = '/content/drive/MyDrive/RoadScan-MA/data/frames_meknes'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(GPX_PATH, 'r') as f:
    gpx = gpxpy.parse(f)

gps_points = []
for track in gpx.tracks:
    for segment in track.segments:
        for p in segment.points:
            gps_points.append({
                'time': p.time, 'lat': p.latitude, 'lon': p.longitude
            })

video_start = datetime(2026, 5, 18, 15, 14, 7, tzinfo=timezone.utc)
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
video_duration = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / fps

def get_gps_at(seconds_elapsed):
    target = video_start + timedelta(seconds=seconds_elapsed)
    for i in range(len(gps_points) - 1):
        t1, t2 = gps_points[i]['time'], gps_points[i+1]['time']
        if t1 <= target <= t2:
            ratio = (target - t1).total_seconds() / (t2 - t1).total_seconds()
            lat = gps_points[i]['lat'] + ratio * (gps_points[i+1]['lat'] - gps_points[i]['lat'])
            lon = gps_points[i]['lon'] + ratio * (gps_points[i+1]['lon'] - gps_points[i]['lon'])
            return round(lat, 8), round(lon, 8)
    return None

INTERVAL, saved, skipped = 2, 0, 0
annotations = []

for sec in range(0, int(video_duration), INTERVAL):
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ret, frame = cap.read()
    if not ret: 
        skipped += 1; continue
    gps = get_gps_at(sec)
    if not gps: 
        skipped += 1; continue

    filename = f"meknes_{saved:04d}.jpg"
    cv2.imwrite(os.path.join(OUTPUT_DIR, filename), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    annotations.append({'filename': filename, 'lat': gps[0], 'lon': gps[1], 'sec': sec})
    saved += 1

cap.release()
with open(os.path.join(OUTPUT_DIR, 'gps_mapping.json'), 'w') as f:
    json.dump(annotations, f, indent=2)
print(f"✅ Extraction terminée : {saved} frames sauvegardées.")

In [ ]:
# ===================================================================
# 2.4 REMAPPING ET GÉNÉRATION DES CROPS POUR CLASSIFICATION
# ===================================================================
CROPS_DIR  = "/content/D1_D40_fixed_crops_v3"
LOCAL_LBLS = "/content/D1_labels_orig"
TRAIN_LIST = "/content/drive/MyDrive/RoadScan-MA/D1_D40_train_files.txt"
os.makedirs(CROPS_DIR, exist_ok=True)

print("⚙️ Préparation du remapping et de l'extraction des bounding boxes...")

## 3. Training (Modèles Intermédiaires et Modèle Final)
Cette section contient les entraînements itératifs. Vous avez utilisé une approche Teacher-Student, en entraînant d'abord des modèles intermédiaires (v1 à v5) sur les niveaux de sévérité D4, puis un classifieur de fissures, et enfin le modèle YOLOv8 final unifié.

In [ ]:
# ===================================================================
# 3.1 ENTRAÎNEMENT DES TEACHERS (SÉVÉRITÉ)
# ===================================================================
from ultralytics import YOLO

print("🚀 Entraînement Teacher V3...")
model_v3 = YOLO('yolov8s.pt')
res_v3 = model_v3.train(
    data='/content/D4_teacher/d4_teacher.yaml',
    epochs=100, imgsz=640, batch=16, name='teacher_v3',
    patience=20, device=0, box=7.5, cls=1.0, dfl=1.5,
    mosaic=1.0, mixup=0.1, scale=0.5, fliplr=0.5,
    optimizer='AdamW', lr0=0.001, warmup_epochs=3,
)

print("🚀 Entraînement Teacher V5...")
model_v5 = YOLO('yolov8s.pt')
res_v5 = model_v5.train(
    data='/content/D4_teacher/data_v5.yaml',
    epochs=200, patience=40, imgsz=640, batch=16, device=0,
    weight_decay=0.0005, lr0=0.001, lrf=0.01, cls=0.5,
    fliplr=0.5, degrees=5.0, hsv_v=0.3, hsv_s=0.3, hsv_h=0.01, mosaic=1.0,
    shear=0.0, scale=0.0, mixup=0.0, copy_paste=0.0, perspective=0.0,
    project="/content/drive/MyDrive/RoadScan-MA/teacher_v5", name="teacher_v5", save=True
)

In [ ]:
# ===================================================================
# 3.2 ENTRAÎNEMENT CLASSIFIEUR DE FISSURES
# ===================================================================
print("\n🚀 Entraînement crack_classifier...")
model_cls = YOLO('yolov8s-cls.pt')

res_cls = model_cls.train(
    data="/content/crack_cls", epochs=150, patience=30, imgsz=224, batch=32, device=0,
    lr0=0.001, cos_lr=True, freeze=5, degrees=45.0, fliplr=0.5, flipud=0.5,
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.4, scale=0.3, close_mosaic=10, weight_decay=0.0005,
    project="/content/drive/MyDrive/RoadScan-MA/crack_classifier", name="crack_clf", save=True
)

In [ ]:
# ===================================================================
# 3.3 ENTRAÎNEMENT DU MODÈLE YOLO UNIFIÉ FINAL
# ===================================================================
DATA_YAML = "/content/RoadScan_YOLO_Final/data.yaml"
SAVE_DIR  = "/content/drive/MyDrive/RoadScan-MA"

print("Lancement entrainement YOLO final...")
model_final = YOLO('yolov8s.pt')

res_final = model_final.train(
    data=DATA_YAML, epochs=100, patience=20, imgsz=640, batch=16, device=0,
    lr0=0.01, lrf=0.001, cos_lr=True, warmup_epochs=3, weight_decay=0.0005,
    multi_scale=True, scale=0.5, mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0,
    degrees=5.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, erasing=0.4,
    dropout=0.0, label_smoothing=0.0, project=SAVE_DIR, name="yolo_final", save=True
)

shutil.copy2(f"{SAVE_DIR}/yolo_final/weights/best.pt", f"{SAVE_DIR}/yolo_final_best.pt")
print("✅ Modèle final sauvegardé avec succès.")

## 4. Results (Évaluation, Inférence et Visualisation)
Validation du modèle final sur le set de test, inférence sur les frames extraites des vidéos réelles, et génération de la cartographie interactive (Folium).

In [ ]:
# ===================================================================
# 4.1 VALIDATION SUR LE JEU DE TEST
# ===================================================================
print("\nEvaluation sur TEST set...")
test_results = model_final.val(
    data=DATA_YAML,
    split='test',
    device=0,
)
print(f"mAP50 TEST : {test_results.results_dict.get('metrics/mAP50(B)',0):.4f}")

In [ ]:
# ===================================================================
# 4.2 GÉNÉRATION DE LA CARTE INTERACTIVE (FOLIUM)
# ===================================================================
import folium

FRAMES_DIR  = '/content/drive/MyDrive/RoadScan-MA/Raw_videos/data/frames_meknes'
GPS_MAPPING = f'{FRAMES_DIR}/gps_mapping.json'
DET_JSON    = '/content/drive/MyDrive/RoadScan-MA/Raw_videos/data/detections_meknes.json'

with open(GPS_MAPPING) as f: 
    gps_data = json.load(f)
with open(DET_JSON) as f: 
    detections = json.load(f)

center_lat = sum(p['lat'] for p in gps_data) / len(gps_data)
center_lon = sum(p['lon'] for p in gps_data) / len(gps_data)
carte = folium.Map(location=[center_lat, center_lon], zoom_start=15)

parcours = [[p['lat'], p['lon']] for p in gps_data]
folium.PolyLine(parcours, color='gray', weight=2, opacity=0.6, tooltip="Parcours filmé").add_to(carte)

COULEURS = {'linear_crack': 'blue', 'alligator_crack': 'orange', 'minor_pothole': 'lightgreen', 'medium_pothole': 'orange', 'major_pothole': 'red'}
ICONES = {'linear_crack': 'minus', 'alligator_crack': 'th', 'minor_pothole': 'circle', 'medium_pothole': 'exclamation-sign', 'major_pothole': 'remove'}

for d in detections:
    folium.Marker(
        location=[d['lat'], d['lon']],
        popup=folium.Popup(f"<b>{d['class']}</b><br>Confiance : {d['conf']:.0%}<br>Frame : {d['filename']}", max_width=200),
        icon=folium.Icon(color=COULEURS.get(d['class'],'gray'), icon=ICONES.get(d['class'],'info-sign'), prefix='glyphicon')
    ).add_to(carte)

carte_path = '/content/drive/MyDrive/RoadScan-MA/carte_meknes_reelle.html'
carte.save(carte_path)
print(f"✅ Carte sauvegardée : {carte_path}")

In [ ]:
# ===================================================================
# 4.3 GÉNÉRATION DES GRAPHIQUES DE DISTRIBUTION
# ===================================================================
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

classes = ['linear_crack', 'alligator_crack', 'minor_pothole', 'medium_pothole', 'major_pothole']
counts  = [3063, 2960, 1997, 2584, 1154]
colors  = ['#4CAF50','#2196F3','#FF9800','#FF5722','#f44336']

bars = ax.bar(classes, counts, color=colors, width=0.6)
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, str(val), ha='center', color='white', fontsize=11)

ax.set_title('Distribution finale — 5 831 images · 11 758 annotations', color='white', fontsize=13, pad=15)
ax.tick_params(colors='white')
for spine in ['top','right']: ax.spines[spine].set_visible(False)
ax.spines['bottom'].set_color('white')
ax.spines['left'].set_color('white')

plt.tight_layout()
plt.savefig('distribution_finale.png', dpi=150, facecolor='#1a1a2e')
print("✅ Graphique de distribution généré (distribution_finale.png).")

## 5. Conclusion (Nettoyage, ZIP Export et Organisation Drive)
Archivage du dataset final au format YOLO, vérification des tailles, et déplacement des fichiers vers les dossiers finaux.

In [ ]:
# ===================================================================
# 5.1 ORGANISATION GOOGLE DRIVE ET ZIP
# ===================================================================
BASE = "/content/drive/MyDrive/RoadScan-MA"
OUT  = f"{BASE}/RoadScan_YOLO_Final"

print("Vérification finale du dataset YOLO...")
for split in ["train","val","test"]:
    ni = len(os.listdir(f"{OUT}/images/{split}"))
    nl = len(os.listdir(f"{OUT}/labels/{split}"))
    ok = "✅" if ni==nl else "❌"
    print(f"  {split:5} : {ni} images | {nl} labels {ok}")

zip_path = f"{BASE}/dataset_yolo"
shutil.make_archive(zip_path, 'zip', OUT)
zip_size = os.path.getsize(f"{zip_path}.zip")/1e9
print(f"✅ ZIP créé : {zip_size:.1f} GB")

print("✅ Pipeline RoadScan-MA terminé avec succès !")